# Functional Testing and Evaluation

This notebook runs a set of ten representative weather-chatbot scenarios and records the observed behaviour.

The tests examine request extraction, location resolution, forecast selection, activity assessment, and response generation. Results should be regenerated against the current application before they are reported as current test evidence.


### Shared project OpenAI client
Run `00_Project_Setup.ipynb` once first. This notebook then uses the shared project configuration; no API key is entered here.

In [5]:
import os
import json
import requests
import pandas as pd

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'config').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from app.services.llm_service import get_client, MODEL_NAME

client = get_client()
print('Shared OpenAI client ready.')


Shared OpenAI client ready.


### Project-level LLM client
This notebook uses the API configuration initialized in `00_Project_Setup.ipynb`. No API key is entered here.

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'config').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from app.services.llm_service import get_client, MODEL_NAME

client = get_client()
print('Shared OpenAI client ready.')


Shared OpenAI client ready.


## 1. Install dependencies

In [3]:
%pip -q install -U openai requests pandas

DEPRECATION: Loading egg at /Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/jupyter-1.0.0-py3.11.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330

[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. Imports and secure API key

Use the Colab Secret named `OPENAI_API_KEY`.

Do not paste the API key into this notebook.

## 3. The 10 test scenarios

In [6]:
TEST_SCENARIOS = [
    {"id": 1, "question": "What's the weather like for my hike in Munich tomorrow?", "expected_activity": "hiking"},
    {"id": 2, "question": "Will it rain during my cycling tour in Berlin tomorrow afternoon?", "expected_activity": "cycling"},
    {"id": 3, "question": "What's the weather for mountain biking in Garmisch-Partenkirchen Saturday morning?", "expected_activity": "mountain biking"},
    {"id": 4, "question": "When would be the best time for running in Cologne tomorrow?", "expected_activity": "running"},
    {"id": 5, "question": "What's the weather like for camping at Lake Garda this weekend, especially overnight?", "expected_activity": "camping"},
    {"id": 6, "question": "I'm hiking near Innsbruck at about 2,500m tomorrow. Will it be colder and windier?", "expected_activity": "hiking"},
    {"id": 7, "question": "Will there be strong wind or rain for my cycling trip in the Black Forest tomorrow?", "expected_activity": "cycling"},
    {"id": 8, "question": "What's the weather for a trail run in the Dolomites Sunday? Could there be rain or snow?", "expected_activity": "trail running"},
    {"id": 9, "question": "Will it be clear for my hike in Salzburg Saturday morning?", "expected_activity": "hiking"},
    {"id": 10, "question": "What's the best day and time for an outdoor tour in Hamburg this weekend?", "expected_activity": "outdoor tour"}
]

pd.DataFrame(TEST_SCENARIOS)

,id,question,expected_activity
0,1,What's the weather like for my hike in Munich ...,hiking
1,2,Will it rain during my cycling tour in Berlin ...,cycling
2,3,What's the weather for mountain biking in Garm...,mountain biking
3,4,When would be the best time for running in Col...,running
4,5,What's the weather like for camping at Lake Ga...,camping
5,6,"I'm hiking near Innsbruck at about 2,500m tomo...",hiking
6,7,Will there be strong wind or rain for my cycli...,cycling
7,8,What's the weather for a trail run in the Dolo...,trail running
8,9,Will it be clear for my hike in Salzburg Satur...,hiking
9,10,What's the best day and time for an outdoor to...,outdoor tour


## 4. Weather Tool

In [7]:
def geocode_location(location):
    url = "https://geocoding-api.open-meteo.com/v1/search"
    params = {"name": location, "count": 1, "language": "en", "format": "json"}

    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()
    data = response.json()

    if "results" not in data or not data["results"]:
        raise ValueError(f"Location not found: {location}")

    result = data["results"][0]

    return {
        "name": result["name"],
        "latitude": result["latitude"],
        "longitude": result["longitude"],
        "country": result.get("country"),
        "timezone": result.get("timezone", "auto")
    }

WEATHER_CODES = {
    0: "Clear sky", 1: "Mainly clear", 2: "Partly cloudy", 3: "Overcast",
    45: "Fog", 48: "Depositing rime fog",
    51: "Light drizzle", 53: "Moderate drizzle", 55: "Dense drizzle",
    61: "Slight rain", 63: "Moderate rain", 65: "Heavy rain",
    71: "Slight snow", 73: "Moderate snow", 75: "Heavy snow",
    80: "Slight rain showers", 81: "Moderate rain showers", 82: "Violent rain showers",
    85: "Slight snow showers", 86: "Heavy snow showers",
    95: "Thunderstorm", 96: "Thunderstorm with slight hail", 99: "Thunderstorm with heavy hail"
}

def get_weather_forecast(latitude, longitude, timezone="auto", forecast_days=7):
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": ",".join([
            "temperature_2m", "precipitation_probability", "precipitation",
            "weather_code", "cloud_cover", "wind_speed_10m"
        ]),
        "timezone": timezone,
        "forecast_days": forecast_days,
        "temperature_unit": "celsius",
        "wind_speed_unit": "kmh",
        "precipitation_unit": "mm"
    }

    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()
    return response.json()

def weather_code_to_description(code):
    return WEATHER_CODES.get(int(code), "Unknown weather condition")

## 4a. Location normalization

The first test revealed two interface failures: natural-language wrappers such as `near` and `the` were passed directly to the geocoder. This small normalization step improves robustness without changing the application architecture.

In [8]:
def normalize_location(location):
    """
    Normalize common natural-language wrappers before geocoding.

    Examples:
    - "near Innsbruck" -> "Innsbruck"
    - "the Dolomites" -> "Dolomites"

    This is intentionally small and transparent for the POC.
    """
    cleaned = str(location).strip()

    prefixes = [
        "near ",
        "around ",
        "in ",
        "at ",
        "the "
    ]

    changed = True
    while changed:
        changed = False
        lower = cleaned.lower()

        for prefix in prefixes:
            if lower.startswith(prefix):
                cleaned = cleaned[len(prefix):].strip()
                changed = True
                break

    return cleaned

# Quick examples
for example in ["near Innsbruck", "the Dolomites", "Munich"]:
    print(f"{example!r} -> {normalize_location(example)!r}")

'near Innsbruck' -> 'Innsbruck'
'the Dolomites' -> 'Dolomites'
'Munich' -> 'Munich'


## 5. Activity Weather Logic

In [9]:
ACTIVITY_PROFILES = {
    "hiking": {"max_rain_probability": 60, "max_wind_kmh": 35, "low_temp_c": 0, "high_temp_c": 30},
    "cycling": {"max_rain_probability": 50, "max_wind_kmh": 30, "low_temp_c": 2, "high_temp_c": 32},
    "mountain biking": {"max_rain_probability": 45, "max_wind_kmh": 30, "low_temp_c": 0, "high_temp_c": 28},
    "running": {"max_rain_probability": 50, "max_wind_kmh": 30, "low_temp_c": 0, "high_temp_c": 30},
    "trail running": {"max_rain_probability": 45, "max_wind_kmh": 30, "low_temp_c": 0, "high_temp_c": 28},
    "camping": {"max_rain_probability": 50, "max_wind_kmh": 25, "low_temp_c": 5, "high_temp_c": 30},
    "outdoor tour": {"max_rain_probability": 50, "max_wind_kmh": 30, "low_temp_c": 2, "high_temp_c": 32}
}

def assess_activity_weather(activity, temperature, wind_speed, precipitation_probability, weather_description=None):
    activity_key = activity.lower().strip()
    if activity_key not in ACTIVITY_PROFILES:
        raise ValueError(f"Unsupported activity: {activity}")

    profile = ACTIVITY_PROFILES[activity_key]
    concerns = []

    if temperature < profile["low_temp_c"]:
        concerns.append("low temperature")
    if temperature > profile["high_temp_c"]:
        concerns.append("high temperature")
    if wind_speed > profile["max_wind_kmh"]:
        concerns.append("strong wind")
    if precipitation_probability > profile["max_rain_probability"]:
        concerns.append("high precipitation probability")

    return {
        "activity": activity_key,
        "temperature_c": round(temperature, 1),
        "wind_kmh": round(wind_speed, 1),
        "precipitation_probability": round(precipitation_probability, 1),
        "weather_description": weather_description,
        "concerns": concerns,
        "summary": (
            "Main factors: " + ", ".join(concerns)
            if concerns else "No major weather concern identified by the prototype rules."
        )
    }

## 6. LLM request extraction

For this evaluation notebook, the LLM also returns `date_iso` using today's date. This makes relative-date scenarios such as Saturday and Sunday testable without hard-coding a calendar date.

In [11]:
from datetime import date

TODAY = date.today()

print("Today's date:", TODAY)

Today's date: 2026-09-24


In [12]:
REQUEST_SCHEMA = {
    "type": "object",
    "properties": {
        "activity": {"type": "string"},
        "location": {"type": "string"},
        "date": {"type": "string"},
        "date_iso": {"type": "string"},
        "time_period": {"type": "string"},
        "elevation_m": {"type": ["number", "null"]}
    },
    "required": ["activity", "location", "date", "date_iso", "time_period", "elevation_m"],
    "additionalProperties": False
}

SYSTEM_PROMPT = f"""
You are the request-understanding component of a weather forecast chatbot.

Today's date is {TODAY.isoformat()}.

Extract:
- activity
- location
- date as stated by the user
- date_iso as YYYY-MM-DD
- time period
- elevation in metres when explicitly provided

Supported activities:
hiking, cycling, mountain biking, running, camping,
trail running, outdoor tour.

Rules:
1. Extract only information supported by the user's wording.
2. Do not invent missing values.
3. Resolve relative dates using today's date.
4. For "Saturday" or "Sunday", return the next occurrence of that weekday.
5. For "this weekend", use the upcoming Saturday as date_iso.
6. If no time period is given, return an empty string.
7. If no elevation is given, return null.
8. Return only the requested structured data.
"""

def extract_weather_request(user_question):
    response = client.responses.create(
        model=MODEL_NAME,
        instructions=SYSTEM_PROMPT,
        input=user_question,
        text={
            "format": {
                "type": "json_schema",
                "name": "weather_request",
                "schema": REQUEST_SCHEMA,
                "strict": True
            }
        }
    )
    return json.loads(response.output_text)

## 7. Forecast selection and aggregation

In [13]:
def select_forecast_period(forecast_df, date_iso, time_period):
    df = forecast_df.copy()
    df["datetime"] = pd.to_datetime(df["time"])

    target_date = pd.to_datetime(date_iso).date()
    df = df[df["datetime"].dt.date == target_date]

    period = time_period.lower().strip()

    if period == "morning":
        df = df[df["datetime"].dt.hour.between(6, 11)]
    elif period == "afternoon":
        df = df[df["datetime"].dt.hour.between(12, 17)]
    elif period == "evening":
        df = df[df["datetime"].dt.hour.between(18, 21)]
    elif period == "overnight":
        df = df[(df["datetime"].dt.hour >= 22) | (df["datetime"].dt.hour <= 5)]

    return df

def summarize_forecast(selected_df):
    if selected_df.empty:
        return None

    return {
        "temperature_min_c": round(float(selected_df["temperature_2m"].min()), 1),
        "temperature_max_c": round(float(selected_df["temperature_2m"].max()), 1),
        "wind_max_kmh": round(float(selected_df["wind_speed_10m"].max()), 1),
        "precipitation_probability_max": round(float(selected_df["precipitation_probability"].max()), 1),
        "precipitation_total_mm": round(float(selected_df["precipitation"].sum()), 1),
        "conditions": list(dict.fromkeys(
            weather_code_to_description(code)
            for code in selected_df["weather_code"].astype(int)
        ))
    }

## 8. LLM response generation

In [14]:
def generate_weather_response(user_question, request, location, forecast_summary, activity_assessment):
    prompt = f"""
User question:
{user_question}

Structured request:
{json.dumps(request, ensure_ascii=False)}

Location:
{json.dumps(location, ensure_ascii=False)}

Forecast:
{json.dumps(forecast_summary, ensure_ascii=False)}

Activity assessment:
{json.dumps(activity_assessment, ensure_ascii=False)}

Write a concise weather chatbot response.

Requirements:
- Use only the supplied forecast for factual weather values.
- Mention temperature, conditions and wind.
- Mention precipitation when relevant.
- Address the user's activity.
- If the user asks for "best time/day", only compare information actually supplied.
- Do not claim guaranteed safety.
- Do not invent missing information.
"""

    response = client.responses.create(
        model=MODEL_NAME,
        instructions=(
            "You generate concise, factual answers for an outdoor weather "
            "chatbot. Ground every factual weather statement in the supplied data."
        ),
        input=prompt
    )
    return response.output_text

## 9. Run one scenario first

In [15]:
def run_single_scenario(scenario):
    question = scenario["question"]
    request = extract_weather_request(question)

    # Normalize the LLM-produced location before sending it to the geocoder.
    original_location = request["location"]
    request["location"] = normalize_location(original_location)

    location = geocode_location(request["location"])

    weather_data = get_weather_forecast(
        latitude=location["latitude"],
        longitude=location["longitude"],
        timezone=location["timezone"],
        forecast_days=7
    )

    forecast_df = pd.DataFrame(weather_data["hourly"])

    selected_df = select_forecast_period(
        forecast_df,
        request["date_iso"],
        request["time_period"]
    )

    forecast_summary = summarize_forecast(selected_df)

    if forecast_summary is None:
        raise ValueError(
            f"No forecast data for requested date {request['date_iso']}."
        )

    average_temperature = (
        forecast_summary["temperature_min_c"]
        + forecast_summary["temperature_max_c"]
    ) / 2

    assessment = assess_activity_weather(
        request["activity"],
        average_temperature,
        forecast_summary["wind_max_kmh"],
        forecast_summary["precipitation_probability_max"],
        ", ".join(forecast_summary["conditions"])
    )

    answer = generate_weather_response(
        question, request, location, forecast_summary, assessment
    )

    return {
        "request": request,
        "original_location": original_location,
        "location": location,
        "forecast": forecast_summary,
        "assessment": assessment,
        "answer": answer
    }

## 10. Run all 10 scenarios

In [16]:
evaluation_rows = []
detailed_results = {}

for scenario in TEST_SCENARIOS:
    scenario_id = scenario["id"]
    question = scenario["question"]

    try:
        result = run_single_scenario(scenario)
        detailed_results[scenario_id] = result

        extracted_activity = result["request"]["activity"]
        activity_match = (
            extracted_activity.lower().strip()
            == scenario["expected_activity"].lower().strip()
        )

        evaluation_rows.append({
            "Scenario": scenario_id,
            "Activity extracted": extracted_activity,
            "Expected activity": scenario["expected_activity"],
            "Activity match": activity_match,
            "Original location": result["original_location"],
            "Normalized location": result["request"]["location"],
            "Geocoded location": result["location"]["name"],
            "Date": result["request"]["date_iso"],
            "Time period": result["request"]["time_period"],
            "Forecast retrieved": True,
            "Response generated": True,
            "Status": "PASS" if activity_match else "CHECK"
        })

    except Exception as e:
        evaluation_rows.append({
            "Scenario": scenario_id,
            "Activity extracted": "",
            "Expected activity": scenario["expected_activity"],
            "Activity match": False,
            "Original location": "",
            "Normalized location": "",
            "Geocoded location": "",
            "Date": "",
            "Time period": "",
            "Forecast retrieved": False,
            "Response generated": False,
            "Status": "ERROR",
            "Error": str(e)
        })

evaluation_df = pd.DataFrame(evaluation_rows)
evaluation_df

,Scenario,Activity extracted,Expected activity,Activity match,Original location,Normalized location,Geocoded location,Date,Time period,Forecast retrieved,Response generated,Status
0,1,hiking,hiking,True,Munich,Munich,Munich,2026-09-25,,True,True,PASS
1,2,cycling,cycling,True,Berlin,Berlin,Berlin,2026-09-25,afternoon,True,True,PASS
2,3,mountain biking,mountain biking,True,Garmisch-Partenkirchen,Garmisch-Partenkirchen,Garmisch-Partenkirchen,2026-09-26,morning,True,True,PASS
3,4,running,running,True,Cologne,Cologne,Cologne,2026-09-25,,True,True,PASS
4,5,camping,camping,True,Lake Garda,Lake Garda,Lake Garda Dam,2026-09-26,overnight,True,True,PASS
5,6,hiking,hiking,True,near Innsbruck,Innsbruck,Innsbruck,2026-09-25,,True,True,PASS
6,7,cycling,cycling,True,Black Forest,Black Forest,Black Forest,2026-09-25,,True,True,PASS
7,8,trail running,trail running,True,Dolomites,Dolomites,Dolomites,2026-09-27,,True,True,PASS
8,9,hiking,hiking,True,Salzburg,Salzburg,Salzburg,2026-09-26,morning,True,True,PASS
9,10,outdoor tour,outdoor tour,True,Hamburg,Hamburg,Hamburg,2026-09-26,,True,True,PASS


## 11. Quantitative test summary

In [18]:
total = len(evaluation_df)
passed = int((evaluation_df["Status"] == "PASS").sum())
check = int((evaluation_df["Status"] == "CHECK").sum())
errors = int((evaluation_df["Status"] == "ERROR").sum())

print(f"Total scenarios tested: {total}")
print(f"PASS: {passed}")
print(f"CHECK: {check}")
print(f"ERROR: {errors}")
print(f"Technical completion rate: {(passed + check) / total * 100:.1f}%")
print(f"Activity extraction match rate: {passed / total * 100:.1f}%")

Total scenarios tested: 10
PASS: 10
CHECK: 0
ERROR: 0
Technical completion rate: 100.0%
Activity extraction match rate: 100.0%


## 12. Inspect generated answers

In [19]:
for scenario_id, result in detailed_results.items():
    print("=" * 80)
    print(f"SCENARIO {scenario_id}")
    print("REQUEST:", TEST_SCENARIOS[scenario_id - 1]["question"])
    print("ANSWER:")
    print(result["answer"])
    print()

SCENARIO 1
REQUEST: What's the weather like for my hike in Munich tomorrow?
ANSWER:
For your hike in Munich tomorrow (2026-09-25):

- Temperature: low 5.8°C, high 18.3°C (activity-assessed ~12.1°C).  
- Conditions: clear sky → mainly clear → partly cloudy through the day.  
- Wind: light, max ~6.2 km/h.  
- Precipitation: 0.0 probability, 0.0 mm (no precipitation expected).

This forecast indicates good hiking weather with no major weather concerns identified by the prototype rules.

SCENARIO 2
REQUEST: Will it rain during my cycling tour in Berlin tomorrow afternoon?
ANSWER:
Tomorrow afternoon in Berlin: temperature about 14.7°C (day range 12.8–16.6°C), overcast skies, wind up to 10.4 km/h. Precipitation probability 0.0 and 0.0 mm expected — no rain forecast. For cycling: conditions look suitable and no major weather concerns are identified by the supplied forecast, but no guarantee of safety.

SCENARIO 3
REQUEST: What's the weather for mountain biking in Garmisch-Partenkirchen Saturd

## 13. Save evaluation results

In [20]:
output_file = "weather_chatbot_evaluation.csv"
evaluation_df.to_csv(output_file, index=False)
print(f"Saved: {output_file}")

Saved: weather_chatbot_phase2_evaluation.csv


## 13a. Iteration from the first evaluation

The first evaluation achieved **8/10 completed scenarios**. The two errors were caused by natural-language location wrappers (`near Innsbruck` and `the Dolomites`) being passed directly to the geocoder. Version 2 adds a transparent normalization step before geocoding. The same 10 scenarios should now be rerun to measure the effect of this change.

## 14. Limitations to record

- The prototype depends on Open-Meteo for factual forecast data.
- Activity thresholds are engineering heuristics for the prototype, not authoritative safety thresholds.
- Elevation is extracted from the request, but this prototype does not calculate a separate elevation-adjusted forecast.
- "Best time/day" questions require stronger comparison logic than a single forecast-window summary.
- Natural-language date handling remains a prototype component.
- Successful API/LLM execution does not establish real-world forecast accuracy or guarantee safety.
- The 10 scenarios are representative initial-design tests, not a statistically representative benchmark dataset.

## 15. Testing summary

After this notebook, we have a systematic test harness for the 10 defined test scenarios.

### Next step: Notebook 06 — Streamlit prototype

We will put the working pipeline behind a simple chatbot interface so the project can be demonstrated as an actual application rather than only as notebook cells.